In [1]:
# try to mount google colab
import os
try:
  from google.colab import drive
  drive.mount('/content/drive')
  cur_dir = "/content/drive/MyDrive/ws/prj/adv_ds_2310/Lab05_Models/ex01_iris_classification/preprocessing"
except:
  cur_dir = "."
  pass
cur_dir = os.path.abspath(cur_dir)
os.chdir(cur_dir)

print(f'cur_dir: {cur_dir} \n --> {os.path.abspath(".")}')

cur_dir: c:\Users\ARtone\Documents\ai_pratice_prj\sgu2526k1_csttnt-main\NhomCSTTNT25-26\DuAn\project\preprocessing 
 --> c:\Users\ARtone\Documents\ai_pratice_prj\sgu2526k1_csttnt-main\NhomCSTTNT25-26\DuAn\project\preprocessing


#### Khai báo thư viện

In [2]:
# Load libraries
import os, sys
from IPython import display
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import joblib
from scipy.stats import chi2_contingency
import math
from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, KFold
from itertools import combinations

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    auc
)
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

import warnings

%matplotlib inline
# plt.rcParams["figure.figsize"] = (12, 6)
# plt.rcParams['figure.dpi'] = 100

warnings.filterwarnings("ignore")

# 3. Tiền xử lí dữ liệu

In [3]:
exps_dir = "../../exps"
if os.path.exists(exps_dir) == False: # tạo thư mục (nếu chưa có)
  os.makedirs(exps_dir, exist_ok=True)

save_dir = f"{exps_dir}/feature1"
os.makedirs(save_dir, exist_ok=True)

In [4]:
test_size=0.3
seed=42
df_dataset=pd.read_excel(f'{exps_dir}/data/data_EDA.xlsx')
df_dataset

,variance,skewness,curtosis,entropy,class
0,3.62160,8.66610,-2.8073,-0.44699,0
1,4.54590,8.16740,-2.4586,-1.46210,0
2,3.86600,-2.63830,1.9242,0.10645,0
3,3.45660,9.52280,-4.0112,-3.59440,0
4,0.32924,-4.45520,4.5718,-0.98880,0
...,...,...,...,...,...
1343,0.40614,1.34920,-1.4501,-0.55949,1
1344,-1.38870,-4.87730,6.4774,0.34179,1
1345,-3.75030,-13.45860,17.5932,-2.77710,1
1346,-3.56370,-8.38270,12.3930,-1.28230,1


### Chia dữ liệu thực nghiệm

+ Dữ liệu ban đầu: chia 30% dữ liệu dùng để Test, 70% dùng để train

In [5]:
data_train, data_test = train_test_split(df_dataset, test_size = 0.3, random_state=seed)
print(data_train.shape,data_test.shape)
data_train.to_excel(f'{exps_dir}/data/datatrain.xlsx', index=None)
data_test.to_excel(f'{exps_dir}/data/datatest.xlsx', index=None)

(943, 5) (405, 5)


In [6]:
data_train

,variance,skewness,curtosis,entropy,class
1050,-0.41965,2.9094,-1.78590,-2.206900,1
171,-1.11930,10.7271,2.09380,-5.650400,0
1171,-1.97250,2.8825,-2.30860,-2.372400,1
1236,-3.00000,-9.1566,9.57660,-0.730180,1
260,1.59400,4.7055,1.37580,0.081882,0
...,...,...,...,...,...
1095,-1.43750,-1.8624,4.02600,0.551270,1
1130,-0.36025,-4.4490,2.10670,0.943080,1
1294,-0.49281,3.0605,-1.83560,-2.834000,1
860,-1.01120,2.9984,-1.16640,-1.618500,1


In [7]:
numeric_columns=dict(np.load(f'{exps_dir}/data/columns_dtype.npz',allow_pickle=True))['numeric_columns']
numeric_columns

array(['variance', 'skewness', 'curtosis', 'entropy'], dtype='<U8')

In [8]:
category_columns=dict(np.load(f'{exps_dir}/data/columns_dtype.npz',allow_pickle=True))['category_columns']
category_columns

array(['class'], dtype='<U5')

In [9]:
(data_train=='?').sum()

variance    0
skewness    0
curtosis    0
entropy     0
class       0
dtype: int64

#### Nhận xét:
+ Không có dữ liệu nào bị lỗi bởi giá trị '?'

## 4. Chuẩn hóa dữ liệu

In [10]:
label_encoders = {}
for column in category_columns:
    if column != "class":
        label_encoder = LabelEncoder()
        data_train[column] = label_encoder.fit_transform(data_train[column])
        label_encoders[column] = label_encoder  # Lưu trữ label encoder nếu cần sau này

scaler = StandardScaler()
data_train[numeric_columns] = scaler.fit_transform(data_train[numeric_columns])
data_train

,variance,skewness,curtosis,entropy,class
1050,-0.311867,0.151033,-0.723971,-0.470361,1
171,-0.556687,1.488346,0.169771,-2.091631,0
1171,-0.855236,0.146432,-0.844382,-0.548281,1
1236,-1.214777,-1.913003,1.893536,0.224909,1
260,0.392745,0.458278,0.004370,0.607245,0
...,...,...,...,...,...
1095,-0.668030,-0.665241,0.614880,0.828242,1
1130,-0.291082,-1.107711,0.172743,1.012714,1
1294,-0.337467,0.176881,-0.735420,-0.765612,1
860,-0.518861,0.166258,-0.581260,-0.193330,1


In [11]:
x_train = data_train.drop('class', axis=1)
y_train = data_train['class']

In [12]:
from imblearn.over_sampling import SMOTE
x_train, y_train = SMOTE().fit_resample(x_train, y_train)
x_train.to_excel(f'{save_dir}/x_train.xlsx', index=False)
y_train.to_excel(f'{save_dir}/y_train.xlsx', index=False)

In [13]:
y_train.value_counts()

class
1    517
0    517
Name: count, dtype: int64

#### Nhận xét:
+ Trọng số của class được cân bằng nên không cần tính trọng số cho từng class khi dữ liệu bị mất cân bằng

In [14]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

print(f"x_train size: {len(x_train)}")

for fold, (train_idx, valid_idx) in enumerate(kfold.split(x_train, y_train)):
    print(f"\nFold {fold+1}")
    print("Train size:", len(train_idx))
    print("Valid size:", len(valid_idx))

    x_tr = x_train.iloc[train_idx]
    x_val = x_train.iloc[valid_idx]
    y_tr = y_train.iloc[train_idx]
    y_val = y_train.iloc[valid_idx]

    model = LogisticRegression(max_iter=1000)
    model.fit(x_tr, y_tr)

    score = model.score(x_val, y_val)
    print("Validation accuracy:", round(score, 3))

x_train size: 1034

Fold 1
Train size: 827
Valid size: 207
Validation accuracy: 0.995

Fold 2
Train size: 827
Valid size: 207
Validation accuracy: 0.99

Fold 3
Train size: 827
Valid size: 207
Validation accuracy: 0.995

Fold 4
Train size: 827
Valid size: 207
Validation accuracy: 0.986

Fold 5
Train size: 828
Valid size: 206
Validation accuracy: 0.99


In [15]:
object_cols_test = dict(
    np.load(f'{exps_dir}/data/columns_dtype.npz', allow_pickle=True)
)['category_columns']

numeric_cols_test = dict(
    np.load(f'{exps_dir}/data/columns_dtype.npz', allow_pickle=True)
)['numeric_columns']


# encode categorical (nếu có)
label_encoders = {}
for column in object_cols_test:
    label_encoder = LabelEncoder()
    data_test[column] = label_encoder.fit_transform(data_test[column])
    label_encoders[column] = label_encoder


# scale numeric
scaler = StandardScaler()
data_test[numeric_cols_test] = scaler.fit_transform(data_test[numeric_cols_test])


# tách feature và label
x_test = data_test.drop('class', axis=1)
y_test = data_test['class']


# lưu file
data_test.to_excel(f'{save_dir}/data_test_minmax.xlsx', index=False)
x_test.to_excel(f'{save_dir}/x_test.xlsx', index=False)
y_test.to_excel(f'{save_dir}/y_test.xlsx', index=False)